<a href="https://colab.research.google.com/github/bhashitha2301/bhashitha2301/blob/main/Textsummary.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.tokenize import sent_tokenize

In [ ]:
!pip install transformers
!pip install datasets

In [ ]:
from datasets import load_dataset

# Load the dataset (use 'test' split for examples)
dataset = load_dataset("cnn_dailymail", '3.0.0', split='test[:5]')  # Load only 5 for demo

# View one example
for i, sample in enumerate(dataset):
    print(f"\n🔹 Document {i+1}:\n", sample['article'][:500], "...")
    print(f"\n🔸 Reference Summary:\n", sample['highlights'], "\n")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


ValueError: Invalid pattern: '**' can only be an entire path component

In [ ]:
from transformers import pipeline

# Load the BART summarizer
summarizer = pipeline("summarization", model="facebook/bart-large-cnn")

In [ ]:
for i, sample in enumerate(dataset):
    article = sample['article']

    # Optional: truncate if text is too long (BART has ~1024 token limit)
    input_text = article[:1024]

    summary = summarizer(input_text, max_length=130, min_length=30, do_sample=False)

    print(f"\n🔹 Document {i+1}:\n", input_text[:500], "...")
    print(f"\n🔸 Generated Summary:\n", summary[0]['summary_text'])
    print(f"\n📌 Reference Summary:\n", sample['highlights'])
    print("-" * 100)

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
!pip install PyMuPDF

In [ ]:
import fitz  # PyMuPDF

filename = list(uploaded.keys())[0]

# Read PDF
doc = fitz.open(filename)
text = ""
for page in doc:
    text += page.get_text()

# Truncate if too long
text = text[:3000]  # Adjust based on model token limit
print("📄 Document Preview:\n", text[:500], "...")

In [ ]:
from transformers import pipeline

# Load summarizer
summarizer = pipeline("summarization", model="facebook/bart-large-cnn")

# Summarize
summary = summarizer(text, max_length=130, min_length=30, do_sample=False)

print("\n🔸 Generated Summary:\n", summary[0]['summary_text'])

In [ ]:
!pip install gradio

In [ ]:
import gradio as gr
from transformers import pipeline
import fitz  # PyMuPDF for PDFs

# Load summarizer
summarizer = pipeline("summarization", model="facebook/bart-large-cnn")

def summarize_file(file):
    # Read file contents
    file_type = file.name.split('.')[-1].lower()

    if file_type == 'txt':
        text = file.read().decode('utf-8')
    elif file_type == 'pdf':
        doc = fitz.open(stream=file.read(), filetype="pdf")
        text = ""
        for page in doc:
            text += page.get_text()
    else:
        return "Unsupported file type. Please upload a TXT or PDF."

    # Truncate text for summarization limit
    text = text[:3000]

    # Generate summary
    summary = summarizer(text, max_length=130, min_length=30, do_sample=False)
    return summary[0]['summary_text']

# Create Gradio interface
iface = gr.Interface(
    fn=summarize_file,
    inputs=gr.File(file_types=[".txt", ".pdf"]),
    outputs="text",
    title="Document Summarizer",
    description="Upload a TXT or PDF file to get an abstractive summary."
)

iface.launch()

In [ ]:
import gradio as gr
from transformers import pipeline
import fitz  # PyMuPDF for PDFs

# Load summarizer
summarizer = pipeline("summarization", model="facebook/bart-large-cnn")

def chunk_text(text, max_chunk=1000):
    """
    Split text into chunks of approximately max_chunk words,
    trying to split at sentence boundaries.
    """
    import re

    sentences = re.split(r'(?<=[.!?]) +', text)
    chunks = []
    current_chunk = []

    current_length = 0
    for sentence in sentences:
        sentence_length = len(sentence.split())
        if current_length + sentence_length > max_chunk:
            chunks.append(" ".join(current_chunk))
            current_chunk = [sentence]
            current_length = sentence_length
        else:
            current_chunk.append(sentence)
            current_length += sentence_length

    if current_chunk:
        chunks.append(" ".join(current_chunk))

    return chunks

def summarize_file(file):
    # Detect file type and extract text
    file_type = file.name.split('.')[-1].lower()

    if file_type == 'txt':
        text = file.read().decode('utf-8')
    elif file_type == 'pdf':
        doc = fitz.open(stream=file.read(), filetype="pdf")
        text = ""
        for page in doc:
            text += page.get_text()
    else:
        return "Unsupported file type. Please upload a TXT or PDF."

    # Chunk text into ~1000 words per chunk (adjust as needed)
    chunks = chunk_text(text, max_chunk=1000)

    # Summarize each chunk
    summaries = []
    for chunk in chunks:
        summary = summarizer(chunk, max_length=130, min_length=30, do_sample=False)
        summaries.append(summary[0]['summary_text'])

    # Combine summaries into a final summary (you can join or re-summarize)
    final_summary = " ".join(summaries)

    return final_summary

# Gradio Interface (same as before)
iface = gr.Interface(
    fn=summarize_file,
    inputs=gr.File(file_types=[".txt", ".pdf"]),
    outputs="text",
    title="Document Summarizer with Chunking",
    description="Upload a long TXT or PDF file to get an abstractive summary. Text longer than 1000 words is chunked."
)

iface.launch()

In [ ]:
import gradio as gr
from transformers import pipeline
import fitz  # PyMuPDF
import re
def summarize_file(file):
    import os
    import fitz  # PyMuPDF
    import re

    try:
        # Check if 'file' has .name attribute (tempfile)
        if hasattr(file, "name"):
            filepath = file.name
        else:
            # fallback: treat 'file' as a string path
            filepath = file

        file_ext = filepath.split('.')[-1].lower()

        if file_ext == 'txt':
            with open(filepath, 'r', encoding='utf-8') as f:
                text = f.read()

        elif file_ext == 'pdf':
            doc = fitz.open(filepath)
            text = ""
            for page in doc:
                text += page.get_text()

        else:
            return "Unsupported file type. Please upload a TXT or PDF."

        if not text.strip():
            return "File appears to be empty or unreadable."

        # Split text into chunks (~1000 words)
        sentences = re.split(r'(?<=[.!?]) +', text)
        chunks = []
        current_chunk = []
        current_length = 0
        max_chunk = 1000
        for sentence in sentences:
            sentence_len = len(sentence.split())
            if current_length + sentence_len > max_chunk:
                chunks.append(" ".join(current_chunk))
                current_chunk = [sentence]
                current_length = sentence_len
            else:
                current_chunk.append(sentence)
                current_length += sentence_len
        if current_chunk:
            chunks.append(" ".join(current_chunk))

        # Summarize each chunk
        summaries = []
        for i, chunk in enumerate(chunks):
            print(f"Summarizing chunk {i+1}/{len(chunks)} ...")
            summary = summarizer(chunk, max_length=130, min_length=30, do_sample=False)
            summaries.append(summary[0]['summary_text'])

        # Combine summaries
        final_summary = " ".join(summaries)
        return final_summary

    except Exception as e:
        return f"An error occurred: {str(e)}"

    sentences = re.split(r'(?<=[.!?]) +', text)
    chunks = []
    current_chunk = []
    current_length = 0
    for sentence in sentences:
        sentence_length = len(sentence.split())
        if current_length + sentence_length > max_chunk:
            chunks.append(" ".join(current_chunk))
            current_chunk = [sentence]
            current_length = sentence_length
        else:
            current_chunk.append(sentence)
            current_length += sentence_length
    if current_chunk:
        chunks.append(" ".join(current_chunk))
    return chunks

def summarize_file(file):
    try:
        file_type = file.name.split('.')[-1].lower()
        if file_type == 'txt':
            text = file.read().decode('utf-8')
        elif file_type == 'pdf':
            doc = fitz.open(stream=file.read(), filetype="pdf")
            text = ""
            for page in doc:
                text += page.get_text()
        else:
            return "Unsupported file type. Please upload a TXT or PDF."

        if not text.strip():
            return "File appears to be empty or unreadable."

        chunks = chunk_text(text, max_chunk=1000)
        summaries = []

        for i, chunk in enumerate(chunks):
            print(f"Summarizing chunk {i+1}/{len(chunks)} ...")
            summary = summarizer(chunk, max_length=130, min_length=30, do_sample=False)
            summaries.append(summary[0]['summary_text'])

        final_summary = " ".join(summaries)
        return final_summary

    except Exception as e:
        return f"An error occurred: {str(e)}"

iface = gr.Interface(
    fn=summarize_file,
    inputs=gr.File(file_types=[".txt", ".pdf"]),
    outputs="text",
    title="Document Summarizer with Debugging",
    description="Upload a long TXT or PDF file to get an abstractive summary."
)

iface.launch()

In [ ]:
# Install dependencies (run once in your Colab or environment)
!pip install gradio transformers PyMuPDF --quiet

import gradio as gr
from transformers import pipeline
import fitz  # PyMuPDF
import re

# Load summarization pipeline (this may take a moment)
summarizer = pipeline("summarization", model="facebook/bart-large-cnn")

def summarize_file(file):
    try:
        if hasattr(file, "name"):
            filepath = file.name
        else:
            filepath = file

        file_ext = filepath.split('.')[-1].lower()

        if file_ext == 'txt':
            with open(filepath, 'r', encoding='utf-8') as f:
                text = f.read()
        elif file_ext == 'pdf':
            doc = fitz.open(filepath)
            text = ""
            for page in doc:
                text += page.get_text()
        else:
            return "Unsupported file type. Please upload a TXT or PDF."

        if not text.strip():
            return "File appears to be empty or unreadable."

        def chunk_text(text, max_chunk=500):  # Reduced chunk size here
            sentences = re.split(r'(?<=[.!?]) +', text)
            chunks = []
            current_chunk = []
            current_length = 0
            for sentence in sentences:
                sentence_len = len(sentence.split())
                if current_length + sentence_len > max_chunk:
                    chunks.append(" ".join(current_chunk))
                    current_chunk = [sentence]
                    current_length = sentence_len
                else:
                    current_chunk.append(sentence)
                    current_length += sentence_len
            if current_chunk:
                chunks.append(" ".join(current_chunk))
            return chunks

        chunks = chunk_text(text, max_chunk=500)

        summaries = []
        for i, chunk in enumerate(chunks):
            print(f"Summarizing chunk {i+1}/{len(chunks)} ...")
            try:
                summary = summarizer(chunk, max_length=130, min_length=30, do_sample=False)
                summaries.append(summary[0]['summary_text'])
            except Exception as e:
                summaries.append(f"[Chunk {i+1} summary failed: {str(e)}]")

        final_summary = " ".join(summaries)
        return final_summary

    except Exception as e:
        return f"An error occurred: {str(e)}"


        # Function to chunk text into ~1000 word chunks at sentence boundaries
        def chunk_text(text, max_chunk=1000):
            sentences = re.split(r'(?<=[.!?]) +', text)
            chunks = []
            current_chunk = []
            current_length = 0
            for sentence in sentences:
                sentence_len = len(sentence.split())
                if current_length + sentence_len > max_chunk:
                    chunks.append(" ".join(current_chunk))
                    current_chunk = [sentence]
                    current_length = sentence_len
                else:
                    current_chunk.append(sentence)
                    current_length += sentence_len
            if current_chunk:
                chunks.append(" ".join(current_chunk))
            return chunks

        chunks = chunk_text(text, max_chunk=1000)

        summaries = []
        for i, chunk in enumerate(chunks):
            print(f"Summarizing chunk {i+1}/{len(chunks)} ...")
            summary = summarizer(chunk, max_length=130, min_length=30, do_sample=False)
            summaries.append(summary[0]['summary_text'])

        final_summary = " ".join(summaries)
        return final_summary

    except Exception as e:
        return f"An error occurred: {str(e)}"

iface = gr.Interface(
    fn=summarize_file,
    inputs=gr.File(file_types=[".txt", ".pdf"]),
    outputs="text",
    title="Document Summarizer with Chunking",
    description="Upload a long TXT or PDF file to get an abstractive summary."
)

iface.launch()

In [ ]:
print(type(file))
print(file)

In [ ]:
!pip install -U fsspec==2023.6.0

In [ ]:
pip install nltk scikit-learn

In [ ]:
import nltk
try:
    nltk.data.find('tokenizers/punkt')
except nltk.downloader.DownloadError:
    nltk.download('punkt')

try:
    nltk.data.find('corpora/stopwords')
except nltk.downloader.DownloadError:
    nltk.download('stopwords')

In [ ]:
import nltk
nltk.download('punkt')
nltk.download('stopwords')

In [ ]:
documents = [
    "This is the first document. It talks about the importance of natural language processing. NLP is a subfield of artificial intelligence that focuses on enabling computers to understand and process human language.",
    "The second document is much shorter. It simply states that the weather today is sunny and warm.",
    "Document number three provides more detailed information about a specific topic. It discusses the latest advancements in deep learning for image recognition, highlighting convolutional neural networks and transformer architectures.",
    "Here is a fourth document. It contains a brief summary of a news article about a recent political event. The article reported on the outcome of the election and the reactions of various political parties."
]

# You can add as many documents as you like to this list.

In [ ]:
import nltk
from nltk.tokenize import sent_tokenize
from nltk.corpus import stopwords
import re
from sklearn.feature_extraction.text import TfidfVectorizer

nltk.download('punkt')
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

def preprocess(text):
    text = re.sub(r'[^\w\s]', '', text).lower()
    tokens = nltk.word_tokenize(text)
    tokens = [word for word in tokens if word not in stop_words]
    return " ".join(tokens)

def split_into_sentences(text):
    return sent_tokenize(text)
def score_sentences(sentences, tfidf_matrix, vectorizer):
    sentence_scores = []
    for i, doc_sentences in enumerate(sentences):
        doc_scores = []
        for sentence in doc_sentences:
            processed_sentence = preprocess(sentence)
            words = nltk.word_tokenize(processed_sentence)
            score = 0
            for word in words:
                if word in vectorizer.vocabulary_:
                    word_index = vectorizer.vocabulary_[word]
                    score += tfidf_matrix[i, word_index]
                doc_scores.append((sentence, score))
            sentence_scores.append(doc_scores)
        return sentence_scores

def get_summary(document_sentences_with_scores, top_n=2): # Adjusted top_n for example
    summary_sentences = sorted(document_sentences_with_scores, key=lambda x: x[1], reverse=True)[:top_n]
    original_indices = {sentence: i for i, sentence in enumerate([s[0] for s in document_sentences_with_scores])}
    summary_sentences = sorted(summary_sentences, key=lambda x: original_indices[x[0]])
    return " ".join([sentence for sentence, score in summary_sentences])


In [ ]:
documents = [
    "This is the first document. It talks about the importance of natural language processing. NLP is a subfield of artificial intelligence that focuses on enabling computers to understand and process human language.",
    "The second document is much shorter. It simply states that the weather today is sunny and warm.",
    "Document number three provides more detailed information about a specific topic. It discusses the latest advancements in deep learning for image recognition, highlighting convolutional neural networks and transformer architectures.",
    "Here is a fourth document. It contains a brief summary of a news article about a recent political event. The article reported on the outcome of the election and the reactions of various political parties."
]

processed_documents = [preprocess(doc) for doc in documents]
sentences = [split_into_sentences(doc) for doc in documents]

vectorizer = TfidfVectorizer()
vectorizer.fit(processed_documents)
tfidf_matrix = vectorizer.transform(processed_documents)

sentence_scores = score_sentences(sentences, tfidf_matrix, vectorizer)
summaries = [get_summary(scores) for scores in sentence_scores]

for i, doc in enumerate(documents):
    print(f"Original Document {i+1}:\n{doc}\n")
    print(f"Summary {i+1}:\n{summaries[i]}\n{'='*50}\n")